In [1]:
import sys
from pathlib import Path
workspace_root = Path().cwd().parent
print(f"Workspace root: {workspace_root}")
sys.path.insert(0, str(workspace_root))

Workspace root: e:\Project\news_secretary\apps\back


In [ ]:
import os
from dotenv import load_dotenv

from apps.back.app.core.pipeline import articles_request_summary_pipeline

e:\Project\news_secretary\apps\back\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [4]:
load_dotenv()

serper_api_key = os.getenv("SERPER_API")
article_summarizer_api_key = os.getenv("ARTICLE_SUMMARIZER_API_KEY", "")

print(f"SERPER_API_KEY: {serper_api_key}")
print(f"ARTICLE_SUMMARIZER_API_KEY: {article_summarizer_api_key}")

SERPER_API_KEY: c765c370cf5cb51ff71c729d78042c5836d4e26a
ARTICLE_SUMMARIZER_API_KEY: ollama


In [5]:

query_list = ["petaling jaya"]

articles_request_summary_pipeline(
    query_list=query_list,
    time_range="1d",
    serper_api_key=serper_api_key,
    article_summarizer_api_key=article_summarizer_api_key,
    keywords_list=query_list,
    language="en",
    region="my"    
    )

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[{'title': '‘New ECRL fares are acceptable, say passengers',
  'link': 'https://www.thestar.com.my/news/nation/2026/10/20/new-ecrl-fares-are-acceptable-say-passengers',
  'snippet': 'PETALING JAYA: The East Coast Rail Link (ECRL) fares are reasonable, as passengers will enjoy shorter travel time, convenience and comfort,...',
  'date': '2026-10-20',
  'source': 'The Star',
  'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRPHJ_8_8xQYrW2Fznm55l79W6PdZOBviePv47yoZFXeLfNp9qSZJNwi6I&usqp=CAI&s',
  'language': 'en',
  'region': 'my',
  'location': None,
  'summary': 'Passengers in Petaling Jaya and surrounding areas have expressed satisfaction with the ECRL fares, citing the shorter travel time, convenience, and comfort offered by the new service. Several individuals, including Najmi Syah and Siti Nur Aisyah, found the fares “acceptable” and valued the reduced travel time, particularly during festive seasons. The ECRL is seen as a viable alternative to buses and driving, 

In [9]:
import sqlite3
from app.configuration import SQLITE_DB_PATH, TABLE_NAMES


def retrieve_leaves_keywords_from_db():
    with sqlite3.connect(SQLITE_DB_PATH) as conn:
        cursor = conn.cursor(row_factory=dict_row)
        cursor.execute(f"SELECT * FROM {TABLE_NAMES['keywords']} WHERE parent_id IS NULL")
        keywords = cursor.fetchall()
    return keywords

def structure_keywords(keywords):
    keyword_dict = {}
    for keyword in keywords:
        # ('medtop:20000893', 'artistic cycling', 'Individual or teams on one or two bicycles in an arena. Points are awarded by a jury.', 'arena cycling , bicycle competition , team cycling', 'medtop:20000892', None, '2026-10-16 06:40:33')
        keyword_id, name, description, synonyms, parent_id, child_ids, created_at = keyword
        keyword_dict[keyword_id] = {
            "keyword": name,
            "description": description,
            "synonyms": synonyms.split(" , ") if synonyms else [],
            "parent_id": parent_id,
            "child_ids": child_ids.split(" , ") if child_ids else [],
            "created_at": created_at
        }
    return keyword_dict

def keywords_structure_to_text(keyword_dict):
    keywords_texts = []
    for _, details in keyword_dict.items():
        keyword_name = details["keyword"]
        keyword_description = details["description"]
        keyword_synonyms = details["synonyms"]
        parents = []
        parent_id = details["parent_id"]
        while parent_id:
            parent_details = keyword_dict.get(parent_id)
            if parent_details:
                parents.append(parent_details["keyword"])
                parent_id = parent_details["parent_id"]
            else:
                break
        
        text = f"Keyword: {keyword_name}\nDescription: {keyword_description}\nSynonyms: {', '.join(keyword_synonyms)}\nParents: {' > '.join(parents[::-1])}\n"
        keywords_texts.append(text)
    return keywords_texts

keywords = retrieve_leaves_keywords_from_db()
keyword_dict = structure_keywords(keywords)
keywords_texts = keywords_structure_to_text(keyword_dict)
keywords_texts[:5]  # Display the first 5 keywords for verification

['Keyword: arts, culture, entertainment and media\nDescription: All forms of arts, entertainment, cultural heritage and media\nSynonyms: Arts, Culture, Entertainment, Media\nParents: \n',
 'Keyword: crime, law and justice\nDescription: The establishment and/or statement of the rules of behaviour in society, the enforcement of these rules, breaches of the rules, the punishment of offenders and the organisations and bodies involved in these activities\nSynonyms: Crime, Law, Justice, Offenses\nParents: \n',
 'Keyword: disaster, accident and emergency incident\nDescription: Man made or natural event resulting in loss of life or injury to living creatures and/or damage to inanimate objects or property\nSynonyms: Disaster, Accident, Emergency, Incidents\nParents: \n',
 'Keyword: economy, business and finance\nDescription: All matters concerning the planning, production and exchange of wealth.\nSynonyms: Economy, Business, Finance, Markets\nParents: \n',
 'Keyword: education\nDescription: All

In [4]:
import requests

from app.configuration import NEWS_SOURCE, SERPER_API


response = requests.post(
    NEWS_SOURCE,
    headers={
        "X-API-KEY": SERPER_API,
        "Content-Type": "application/json"
        },
    json={
        "q": "NVIDIA",
        "tbs": "qdr:d"
    }
)

response.json() , response.status_code

({'message': 'Not enough credits', 'statusCode': 400}, 400)